# 2주차 직접 해보기: 토큰화와 단어 벡터

사이트의 2주차 회독과 정리 슬라이드를 본 다음 풀어요. 위에서부터 차례로 `Shift+Enter` 로 실행해요.

- `# TODO` 가 있는 칸의 `None` 을 알맞은 코드로 바꿔요.
- 바로 아래 **확인 셀**을 실행하면 맞았는지 알려 줘요. 틀리면 빨간 AssertionError 와 힌트가 나와요.
- 막히면 맨 아래 **정답 코드**를 봐요.

GPU 는 필요 없어요. 인터넷 다운로드도 없어요.

In [ ]:
import math
from collections import Counter
def ok(cond, msg):
    assert cond, msg
    print('정답! ' + msg)
print('준비 끝')

## 1. 문장을 토큰으로 자르는 두 가지 방법

강의 N2 p.7(단어 단위), p.9(문자 단위). 공백으로 자르면 **단어(word)** 토큰, 글자 하나하나로 자르면 **문자(character)** 토큰이에요.

`sentence.split()` 은 공백 기준으로 자른 리스트, `list(sentence)` 는 글자 하나씩 자른 리스트를 돌려줘요.

In [ ]:
sentence = "the cat sat on the mat"

words = None   # TODO: 공백으로 자르기
chars = None   # TODO: 글자 하나씩 (공백도 한 글자로 쳐요)

print(words)
print(len(words), len(chars))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(words == ['the', 'cat', 'sat', 'on', 'the', 'mat'], '단어 토큰 6개')
ok(len(chars) == 22, '문자 토큰 22개 (공백 5개 포함)')
ok(len(set(words)) == 5, '서로 다른 단어(어휘)는 5개: the 가 두 번 나와요')

## 2. BPE 1단계: 붙어 있는 쌍 세기

강의 N2 p.11-12 의 장난감 말뭉치예요. 단어를 글자 튜플로 쓰고, 끝에 `</w>`(단어 끝 표시)를 붙였어요. 숫자는 그 단어가 나온 횟수예요.

`count_pairs` 는 **바로 옆에 붙은 두 기호 쌍**이 모두 몇 번 나오는지 세요. 단어가 5번 나왔으면 그 단어 안의 쌍도 5번씩 세요.

In [ ]:
corpus = {
    ('l', 'o', 'w', '</w>'): 5,
    ('l', 'o', 'w', 'e', 'r', '</w>'): 2,
    ('n', 'e', 'w', 'e', 's', 't', '</w>'): 6,
    ('w', 'i', 'd', 'e', 's', 't', '</w>'): 3,
}

def count_pairs(corpus):
    pairs = Counter()
    for word, freq in corpus.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pairs[pair] += None   # TODO: 이 단어가 나온 횟수만큼 더하기
    return pairs

pairs = count_pairs(corpus)
print(pairs.most_common(4))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(pairs[('e', 's')] == 9, "('e','s') 는 newest 6번 + widest 3번 = 9번")
ok(pairs[('l', 'o')] == 7, "('l','o') 는 low 5번 + lower 2번 = 7번")
ok(pairs[('w', 'e')] == 8, "('w','e') 는 lower 2번 + newest 6번 = 8번")

## 3. BPE 2단계: 가장 많은 쌍을 하나로 합치기

`merge` 는 말뭉치의 모든 단어에서 쌍 `(a, b)` 가 붙어 있는 곳을 한 기호 `a + b` 로 바꿔요. 예를 들어 `('e', 's')` 를 합치면 `n e w e s t` 가 `n e w es t` 가 돼요.

In [ ]:
def merge(corpus, pair):
    a, b = pair
    new = {}
    for word, freq in corpus.items():
        out, i = [], 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == a and word[i + 1] == b:
                out.append(None)   # TODO: 두 기호를 이어 붙인 새 기호
                i += 2
            else:
                out.append(word[i])
                i += 1
        new[tuple(out)] = freq
    return new

after = merge(corpus, ('e', 's'))
for w, f in after.items():
    print(' '.join(w), f)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(('n', 'e', 'w', 'es', 't', '</w>') in after, 'newest 가 n e w es t </w> 로 바뀌었어요')
ok(('l', 'o', 'w', '</w>') in after, 'es 가 없는 low 는 그대로예요')

## 4. BPE 학습 반복과 처음 보는 단어 자르기

쌍 세기 → 가장 많은 쌍 합치기 → 규칙 기록을 정해진 횟수만큼 반복하면 BPE 학습이 끝나요(N2 p.11). 배운 병합 규칙을 **같은 순서로** 새 단어에 적용하면 처음 보는 단어도 자를 수 있어요(N2 p.13).

`max(pairs, key=pairs.get)` 은 횟수가 가장 큰 쌍을 골라요(같으면 먼저 센 쌍).

In [ ]:
def train_bpe(corpus, n_merges):
    rules = []
    for _ in range(n_merges):
        pairs = count_pairs(corpus)
        best = None   # TODO: 가장 많이 나온 쌍 고르기
        corpus = merge(corpus, best)
        rules.append(best)
    return rules

def tokenize(word, rules):
    symbols = list(word) + ['</w>']
    for a, b in rules:
        symbols = list(merge({tuple(symbols): 1}, (a, b)))[0]
    return list(symbols)

rules = train_bpe(corpus, 8)
print(rules)
print(tokenize('lowest', rules))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(rules[0] == ('e', 's'), '첫 병합은 9번 나온 (e, s)')
ok(tokenize('lowest', rules) == ['low', 'est</w>'], '학습 때 없던 lowest 도 low + est</w> 로 잘려요')

## 5. Fertility: 단어 하나가 토큰 몇 개로 쪼개지나

강의 N2 p.20. **fertility = 토큰 수 / 공백 단어 수**. 영어 중심 토크나이저는 한국어를 잘게 쪼개서 fertility 가 커져요. 아래 토큰 목록은 설명용으로 만든 **예시**예요(실제 토크나이저 결과는 실습 N2L p.14-15 에서 봐요).

In [ ]:
sentence = '오늘은 날씨가 정말 좋네요'
tokens_a = ['오늘', '은', '날씨', '가', '정말', '좋', '네요']            # 예시: 한국어로 학습한 토크나이저
tokens_b = ['오', '늘', '은', '날', '씨', '가', '정', '말', '좋', '네', '요']  # 예시: 한국어를 잘 모르는 토크나이저

def fertility(tokens, sentence):
    return None   # TODO: 토큰 수 / 공백으로 자른 단어 수

print(fertility(tokens_a, sentence), fertility(tokens_b, sentence))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(fertility(tokens_a, sentence) - 1.75) < 1e-9, '7 토큰 / 4 단어 = 1.75')
ok(abs(fertility(tokens_b, sentence) - 2.75) < 1e-9, '11 토큰 / 4 단어 = 2.75: 토큰이 많을수록 비용과 문맥 자리를 더 써요')

## 6. 내적과 코사인 유사도

강의 N2 p.27, 기초 다지기 2단원. 내적은 **같은 자리끼리 곱해서 모두 더한 값**, 코사인 유사도는 내적을 두 벡터 길이의 곱으로 나눈 값(-1 ~ 1)이에요. 아래 3차원 벡터는 설명용 **장난감 벡터**예요.

In [ ]:
movie = [0.9, 0.8, 0.1]
drama = [0.8, 0.9, 0.2]
kimchi = [0.1, 0.0, 0.9]

def dot(u, v):
    return None   # TODO: 같은 자리끼리 곱해서 더하기 (sum 과 zip 사용)

def norm(u):
    return math.sqrt(dot(u, u))

def cosine(u, v):
    return dot(u, v) / (norm(u) * norm(v))

print(round(cosine(movie, drama), 3), round(cosine(movie, kimchi), 3))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(dot(movie, drama) - 1.46) < 1e-9, '0.9x0.8 + 0.8x0.9 + 0.1x0.2 = 1.46')
ok(cosine(movie, drama) > 0.9 > cosine(movie, kimchi), '영화와 드라마는 가깝고 영화와 김치는 멀어요')

## 7. 소프트맥스: 점수를 확률로

강의 N2 p.33, 기초 다지기 6단원. 점수마다 `exp` 를 씌우고, 전체 합으로 나눠요. 결과는 모두 0보다 크고 합이 1이에요.

In [ ]:
scores = [2.0, 1.0, 0.0]

def softmax(xs):
    exps = [math.exp(x) for x in xs]
    total = None   # TODO: exps 의 합
    return [e / total for e in exps]

probs = softmax(scores)
print([round(p, 3) for p in probs])

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(sum(probs) - 1) < 1e-9, '확률의 합은 1')
ok([round(p, 3) for p in probs] == [0.665, 0.245, 0.09], '가장 큰 점수가 가장 큰 확률을 받아요')

## 8. Skip-gram 이 만드는 (중심 단어, 문맥 단어) 쌍

강의 N2 p.28-29. 윈도우 크기 m 이면 중심 단어 앞뒤 m 칸 안의 단어가 문맥 단어예요. 중심 단어 자기 자신은 빼요.

In [ ]:
tokens = ['나는', '어제', '재미있는', '영화', '를', '봤다']
m = 2

pairs = []
for t, center in enumerate(tokens):
    for j in range(-m, m + 1):
        if j == 0:
            continue
        k = t + j
        if None:   # TODO: k 가 0 이상이고 len(tokens) 보다 작을 때만
            pairs.append((center, tokens[k]))

print(len(pairs))
print([p for p in pairs if p[0] == '영화'])

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(len(pairs) == 18, '6단어, m=2 이면 가장자리를 빼고 18쌍')
ok([p[1] for p in pairs if p[0] == '영화'] == ['어제', '재미있는', '를', '봤다'], '영화의 문맥 단어는 앞 2개, 뒤 2개')

## 9. 단어 유추: 왕 - 남자 + 여자

강의 N2 p.41, p.48. 벡터 더하기 빼기로 만든 점에서 코사인 유사도가 가장 큰 단어를 찾아요(질문에 쓴 단어는 빼요). 아래 2차원 벡터는 **장난감 벡터**예요.

In [ ]:
vec = {
    '왕': [0.9, 0.9], '여왕': [0.9, 0.1], '남자': [0.1, 0.9],
    '여자': [0.1, 0.1], '사과': [0.5, -0.8],
}
target = [vec['왕'][i] - vec['남자'][i] + vec['여자'][i] for i in range(2)]

cands = [w for w in vec if w not in ('왕', '남자', '여자')]
answer = None   # TODO: cands 중 cosine(vec[w], target) 이 가장 큰 w (max 와 key 사용)
print(target, answer)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(answer == '여왕', '왕 - 남자 + 여자 ≈ 여왕')

## 정답 코드

먼저 스스로 풀어 보고, 막혔을 때만 봐요.

**1. 문장을 토큰으로 자르는 두 가지 방법**

```python
sentence = "the cat sat on the mat"

words = sentence.split()
chars = list(sentence)

print(words)
print(len(words), len(chars))
```

**2. BPE 1단계: 붙어 있는 쌍 세기**

```python
corpus = {
    ('l', 'o', 'w', '</w>'): 5,
    ('l', 'o', 'w', 'e', 'r', '</w>'): 2,
    ('n', 'e', 'w', 'e', 's', 't', '</w>'): 6,
    ('w', 'i', 'd', 'e', 's', 't', '</w>'): 3,
}

def count_pairs(corpus):
    pairs = Counter()
    for word, freq in corpus.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pairs[pair] += freq
    return pairs

pairs = count_pairs(corpus)
print(pairs.most_common(4))
```

**3. BPE 2단계: 가장 많은 쌍을 하나로 합치기**

```python
def merge(corpus, pair):
    a, b = pair
    new = {}
    for word, freq in corpus.items():
        out, i = [], 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == a and word[i + 1] == b:
                out.append(a + b)
                i += 2
            else:
                out.append(word[i])
                i += 1
        new[tuple(out)] = freq
    return new

after = merge(corpus, ('e', 's'))
for w, f in after.items():
    print(' '.join(w), f)
```

**4. BPE 학습 반복과 처음 보는 단어 자르기**

```python
def train_bpe(corpus, n_merges):
    rules = []
    for _ in range(n_merges):
        pairs = count_pairs(corpus)
        best = max(pairs, key=pairs.get)
        corpus = merge(corpus, best)
        rules.append(best)
    return rules

def tokenize(word, rules):
    symbols = list(word) + ['</w>']
    for a, b in rules:
        symbols = list(merge({tuple(symbols): 1}, (a, b)))[0]
    return list(symbols)

rules = train_bpe(corpus, 8)
print(rules)
print(tokenize('lowest', rules))
```

**5. Fertility: 단어 하나가 토큰 몇 개로 쪼개지나**

```python
sentence = '오늘은 날씨가 정말 좋네요'
tokens_a = ['오늘', '은', '날씨', '가', '정말', '좋', '네요']
tokens_b = ['오', '늘', '은', '날', '씨', '가', '정', '말', '좋', '네', '요']

def fertility(tokens, sentence):
    return len(tokens) / len(sentence.split())

print(fertility(tokens_a, sentence), fertility(tokens_b, sentence))
```

**6. 내적과 코사인 유사도**

```python
movie = [0.9, 0.8, 0.1]
drama = [0.8, 0.9, 0.2]
kimchi = [0.1, 0.0, 0.9]

def dot(u, v):
    return sum(a * b for a, b in zip(u, v))

def norm(u):
    return math.sqrt(dot(u, u))

def cosine(u, v):
    return dot(u, v) / (norm(u) * norm(v))

print(round(cosine(movie, drama), 3), round(cosine(movie, kimchi), 3))
```

**7. 소프트맥스: 점수를 확률로**

```python
scores = [2.0, 1.0, 0.0]

def softmax(xs):
    exps = [math.exp(x) for x in xs]
    total = sum(exps)
    return [e / total for e in exps]

probs = softmax(scores)
print([round(p, 3) for p in probs])
```

**8. Skip-gram 이 만드는 (중심 단어, 문맥 단어) 쌍**

```python
tokens = ['나는', '어제', '재미있는', '영화', '를', '봤다']
m = 2

pairs = []
for t, center in enumerate(tokens):
    for j in range(-m, m + 1):
        if j == 0:
            continue
        k = t + j
        if 0 <= k < len(tokens):
            pairs.append((center, tokens[k]))

print(len(pairs))
print([p for p in pairs if p[0] == '영화'])
```

**9. 단어 유추: 왕 - 남자 + 여자**

```python
vec = {
    '왕': [0.9, 0.9], '여왕': [0.9, 0.1], '남자': [0.1, 0.9],
    '여자': [0.1, 0.1], '사과': [0.5, -0.8],
}
target = [vec['왕'][i] - vec['남자'][i] + vec['여자'][i] for i in range(2)]

cands = [w for w in vec if w not in ('왕', '남자', '여자')]
answer = max(cands, key=lambda w: cosine(vec[w], target))
print(target, answer)
```